# 06.2 Comparison Operators and Chaining

Six comparison operators, plus a feature almost unique to Python: **chaining**.
`1 < x < 10` is not a quirk — it is a first-class part of the grammar, and it
behaves differently from what most people assume.

**45 numbered examples.**

## Theory

### The six operators

| Operator | Asks | Dunder |
|---|---|---|
| `==` | Equal value? | `__eq__` |
| `!=` | Different value? | `__ne__` |
| `<` | Less than? | `__lt__` |
| `>` | Greater than? | `__gt__` |
| `<=` | Less or equal? | `__le__` |
| `>=` | Greater or equal? | `__ge__` |

All six return `bool` for built-in types — but a class can return anything
(NumPy returns arrays, which is how `array > 5` works).

### Chaining

```python
1 < x < 10
```

expands to:

```python
(1 < x) and (x < 10)
```

with one crucial difference: **`x` is evaluated only once**. That matters when
`x` is an expensive call or has side effects.

Chaining works with any comparison operators, in any combination — including
mixed ones like `a == b < c`, which is legal but usually confusing.

### Comparing different types

Python 3 **refuses** to order unrelated types:

```python
1 < "a"      # TypeError
```

Python 2 allowed it with arbitrary results, which hid real bugs. Equality is
different — `1 == "a"` is simply `False`, no error.

### How sequences compare

Lists, tuples and strings compare **lexicographically**: element by element,
stopping at the first difference. If one runs out first, the shorter one is
smaller.

In [ ]:
# EXAMPLE 1-6: the six operators.
print("EXAMPLE 1-6: the six comparison operators")
print("")

left = 5
right = 3

print(f"   1. {left} == {right}  ->", left == right)
print(f"   2. {left} != {right}  ->", left != right)
print(f"   3. {left} <  {right}  ->", left < right)
print(f"   4. {left} >  {right}  ->", left > right)
print(f"   5. {left} <= {right}  ->", left <= right)
print(f"   6. {left} >= {right}  ->", left >= right)

print("")
print("   All return a real bool:", type(left == right).__name__)

In [ ]:
# EXAMPLE 7-12: comparisons are dunder calls, and can be overridden.
print("EXAMPLE 7-12: comparisons are method calls")
print("")

value = 5
print("   7.  5 == 3          ->", value == 3)
print("       (5).__eq__(3)   ->", value.__eq__(3))
print("")
print("   8.  5 < 3           ->", value < 3)
print("       (5).__lt__(3)   ->", value.__lt__(3))


class Version:
    """A version number that compares by its parts."""

    def __init__(self, text):
        # Split "1.2.3" into a tuple of ints for comparison.
        self.parts = tuple(int(part) for part in text.split("."))
        self.text = text

    def __eq__(self, other):
        return self.parts == other.parts

    def __lt__(self, other):
        # Tuples compare element by element, which is exactly what we want.
        return self.parts < other.parts

    def __repr__(self):
        return f"Version({self.text!r})"


print("")
print("   9.  Custom comparison on a Version class:")
print("       Version('1.2.0') < Version('1.10.0') ->",
      Version("1.2.0") < Version("1.10.0"))
print("       (string comparison would get this WRONG:",
      "'1.2.0' < '1.10.0' ->", "1.2.0" < "1.10.0", ")")

# 10-12. Sorting uses __lt__.
versions = [Version("1.10.0"), Version("1.2.0"), Version("1.9.3")]
print("")
print("   10. sorted() uses __lt__:", sorted(versions))

# functools.total_ordering fills in the rest from __eq__ and __lt__.
print("")
print("   11. We only defined __eq__ and __lt__.")
print("       Version('1.0.0') > Version('0.9.0') ->", end=" ")
try:
    print(Version("1.0.0") > Version("0.9.0"))
except TypeError as error:
    print("TypeError:", error)

print("")
print("   12. Python reflects < into > automatically when possible.")
print("       For the full set, use @functools.total_ordering (Ch27).")

## Chaining

This is the feature worth understanding properly.

In [ ]:
# EXAMPLE 13-18: chained comparisons.
print("EXAMPLE 13-18: chaining")
print("")

age = 25

# 13. The readable form.
print("   13. 18 <= age < 65        ->", 18 <= age < 65)

# 14. What it expands to.
print("   14. equivalent to:", (18 <= age) and (age < 65))

# 15. Three comparisons chained.
print("   15. 0 < age < 100 < 200   ->", 0 < age < 100 < 200)

# 16. Mixed operators are legal.
value = 5
print("   16. 1 < value == 5        ->", 1 < value == 5)

# 17. Equality chains too.
print("   17. 1 == 1 == 1           ->", 1 == 1 == 1)

# 18. Chaining short-circuits: later operands are never evaluated.
def announce(value):
    """Return a value, announcing that it was evaluated."""
    print("       (announce was called)")
    return value


print("   18. short-circuiting - the second call never happens:")
result = 10 < 5 < announce(99)
print("       10 < 5 < announce(99) ->", result)
print("       10 < 5 was False, so the rest was skipped entirely.")

In [ ]:
# EXAMPLE 19-22: the single-evaluation guarantee.
print("EXAMPLE 19-22: each operand is evaluated ONCE")
print("")

call_count = 0


def get_value():
    """Return a value, counting how often it is called."""
    global call_count
    call_count += 1
    return 5


# 19. Chained form - one call.
call_count = 0
result = 1 < get_value() < 10
print("   19. 1 < get_value() < 10")
print("       result:", result, " calls made:", call_count)

# 20. Expanded form - two calls.
call_count = 0
result = (1 < get_value()) and (get_value() < 10)
print("")
print("   20. (1 < get_value()) and (get_value() < 10)")
print("       result:", result, " calls made:", call_count)

print("")
print("   21. This matters when the call is expensive, or has side effects.")

# 22. A real example: consuming from an iterator.
numbers = iter([5, 99])
print("")
print("   22. With an iterator, the difference is visible:")
print("       chained:  1 < next(numbers) < 10 ->", 1 < next(numbers) < 10)
print("       the iterator still holds:", list(numbers))

In [ ]:
# EXAMPLE 23-26: chaining traps.
print("EXAMPLE 23-26: where chaining surprises people")
print("")

# 23. `is` chains too, which is rarely what you want.
value = None
print("   23. value is None is True")
print("       ->", (value is None is True))
print("       expands to (value is None) and (None is True)")
print("       which is False, even though value IS None")

# 24. The != chain is almost never what you mean.
a, b, c = 1, 2, 1
print("")
print("   24. a != b != c  with a=1, b=2, c=1")
print("       ->", a != b != c, "<- says 'all different', but a == c")
print("       For 'all distinct', use len({a, b, c}) == 3 ->",
      len({a, b, c}) == 3)

# 25. Comparing three values for equality DOES work.
print("")
print("   25. a == b == c is correct for 'all equal':")
print("       1 == 1 == 1 ->", 1 == 1 == 1)
print("       1 == 1 == 2 ->", 1 == 1 == 2)

# 26. Readability limit.
print("")
print("   26. Legal but unreadable:")
print("       0 < 5 != 3 <= 10 > 2 ->", 0 < 5 != 3 <= 10 > 2)
print("       Chain two comparisons. Three is pushing it.")

## Comparing different types

In [ ]:
# EXAMPLE 27-32: cross-type comparison.
print("EXAMPLE 27-32: comparing different types")
print("")

# 27-28. Equality across types is allowed and simply returns False.
print("   27. 1 == 'a'     ->", 1 == "a", "<- no error")
print("   28. [1] == (1,)  ->", [1] == (1,), "<- list is never a tuple")

# 29. Numeric types compare correctly across the tower.
print("")
print("   29. numeric comparisons work across types:")
print("       1 == 1.0       ->", 1 == 1.0)
print("       1 == True      ->", 1 == True)
from decimal import Decimal
from fractions import Fraction
print("       Decimal('1') == 1 ->", Decimal("1") == 1)
print("       Fraction(1,2) == 0.5 ->", Fraction(1, 2) == 0.5)

# 30-31. But ORDERING unrelated types is an error.
print("")
print("   30. ordering unrelated types fails:")
for left, right in [(1, "a"), ([1], (1,)), (None, 0)]:
    try:
        left < right
        print(f"       {left!r} < {right!r} -> worked")
    except TypeError as error:
        print(f"       {left!r} < {right!r} -> TypeError: {error}")

# 32. This is a deliberate Python 3 change.
print("")
print("   32. Python 2 allowed this and returned arbitrary results,")
print("       which silently hid real bugs. Python 3 refuses.")

In [ ]:
# EXAMPLE 33-38: sequences compare lexicographically.
print("EXAMPLE 33-38: sequence comparison")
print("")

# 33-34. Element by element, stopping at the first difference.
print("   33. [1, 2, 3] < [1, 2, 4]  ->", [1, 2, 3] < [1, 2, 4])
print("       (first two match, 3 < 4 decides it)")
print("")
print("   34. [1, 2] < [1, 2, 0]     ->", [1, 2] < [1, 2, 0])
print("       (identical so far, shorter one is smaller)")

# 35. Strings use Unicode code points.
print("")
print("   35. string comparison uses code points:")
print("       'apple' < 'banana'  ->", "apple" < "banana")
print("       'Z' < 'a'           ->", "Z" < "a", " (ord: ",
      ord("Z"), "vs", ord("a"), ")")
print("       '10' < '9'          ->", "10" < "9", "<- TEXT, not numbers")

# 36. Which is why sorting version strings fails.
versions = ["1.10.0", "1.9.0", "1.2.0"]
print("")
print("   36. sorting version strings naively:")
print("       ", sorted(versions), "<- 1.10.0 sorts before 1.9.0")
print("       fix: sort by a tuple of ints")
print("       ", sorted(versions, key=lambda v: tuple(int(p) for p in v.split("."))))

# 37. Tuples compare the same way - useful for multi-key sorting.
people = [("Chen", 30), ("Asha", 25), ("Asha", 30)]
print("")
print("   37. tuple comparison sorts by first element, then second:")
print("       ", sorted(people))

# 38. Case-insensitive comparison needs an explicit key.
words = ["banana", "Apple", "cherry"]
print("")
print("   38. case matters by default:")
print("       sorted():             ", sorted(words))
print("       sorted(key=str.lower):", sorted(words, key=str.lower))

## Equality traps

In [ ]:
import math

# EXAMPLE 39-45: where == gives surprising answers.
print("EXAMPLE 39-45: equality traps")
print("")

# 39. Floats.
print("   39. 0.1 + 0.2 == 0.3 ->", 0.1 + 0.2 == 0.3)
print("       use math.isclose ->", math.isclose(0.1 + 0.2, 0.3))

# 40. NaN.
nan = float("nan")
print("")
print("   40. nan == nan ->", nan == nan, "<- never equal to itself")
print("       math.isnan(nan) ->", math.isnan(nan))

# 41. bool and int compare equal, because bool IS an int.
print("")
print("   41. True == 1    ->", True == 1)
print("       True == 1.0  ->", True == 1.0)
print("       [True] == [1] ->", [True] == [1])
print("       Compare values with ==; reserve `is` for None and identity.")

# 42. Different collection types are never equal.
print("")
print("   42. [1,2] == (1,2) ->", [1, 2] == (1, 2), "<- type matters")
print("       {1,2} == [1,2] ->", {1, 2} == [1, 2])
print("       but list(x) == list(y) works:", list((1, 2)) == [1, 2])

# 43. Sets ignore order; sequences do not.
print("")
print("   43. [1,2] == [2,1]   ->", [1, 2] == [2, 1], "<- order matters")
print("       {1,2} == {2,1}   ->", {1, 2} == {2, 1}, "<- order irrelevant")

# 44. == compares values; `is` compares identity.
list_a = [1, 2]
list_b = [1, 2]
print("")
print("   44. two equal lists:")
print("       list_a == list_b ->", list_a == list_b)
print("       list_a is list_b ->", list_a is list_b)

# 45. None must be tested with `is`.
print("")
print("   45. testing for None:")
value = None
print("       value is None ->", value is None, "<- correct")
print("       value == None ->", value == None, "<- works, but E711")

## Takeaways

1. Six comparison operators, all backed by **dunder methods** you can override.
2. **Chaining** `1 < x < 10` expands to `(1 < x) and (x < 10)` but evaluates `x`
   **only once** — which matters for expensive calls and iterators.
3. Chaining works with mixed operators, but `a != b != c` does **not** mean "all
   distinct" — use `len({a, b, c}) == 3`.
4. Python 3 **refuses to order unrelated types**. Equality across types just
   returns `False`.
5. Numeric types compare correctly across the whole tower — `1 == 1.0 == True ==
   Decimal("1")`.
6. Sequences compare **lexicographically**; a shorter prefix is smaller.
7. String comparison uses **code points**, so `'10' < '9'` and `'Z' < 'a'`.
8. Never compare floats with `==`, never compare `nan` with anything, and always
   test `None` with `is`.

## Try it yourself

1. Write a chain that evaluates a function once, and prove it with a counter.
2. Predict `'apple' < 'Banana'`. Check with `ord()`.
3. Sort `["1.10", "1.9", "1.2"]` correctly as versions.
4. Write a class with `__eq__` and `__lt__`, then sort a list of them.
5. Explain why `value is None is True` is `False` when `value` is `None`.